# Evaluating Agents

Companion notebook for the [Evaluating Agents lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/10-evaluating-agents).

**The idea in one sentence.** Agents are **stochastic**, so a single run tells you
almost nothing — you evaluate them over many runs with metrics like **pass@k** (did it
succeed in *any* of $k$ tries) and **consistency** (did it succeed in *all* of them),
and you score both the **outcome** and the **trajectory** (how efficiently it got there).

Two axes:

- **Outcome vs trajectory:** did it solve the task, and how many steps / how much cost
  did it burn getting there.
- **pass@k rises, consistency falls:** more attempts make *some* success more likely
  ($1-(1-p)^k$) but *every*-attempt success less likely ($p^k$).

We build the metrics from scratch and **validate pass@k against Monte-Carlo
simulation**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. Outcome vs trajectory

An agent reaches a goal via a *path*. We model each attempt as: it succeeds with prob `p`, and takes a number of steps (the oracle minimum is `opt`). Efficiency = opt / steps_taken.

In [ ]:
def attempt(p, opt, seed):
    g = np.random.default_rng(seed)
    solved = g.random() < p
    steps = opt + g.poisson(3)  # wandered a few extra steps
    return solved, steps

solved, steps = attempt(p=0.6, opt=5, seed=1)
print('solved:', solved, ' steps:', steps, ' efficiency:', round(5/steps, 2))

## 2. A single run lies — use pass@k and consistency

For a task an agent solves with probability `p` per attempt:
- **pass@k** = probability of solving in at least one of k tries = $1-(1-p)^k$.
- **consistency** (pass^k) = solving in *all* k tries = $p^k$.

A capable-but-flaky agent has high pass@k but low consistency.

In [ ]:
ks = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for p in [0.3, 0.6]:
    ax.plot(ks, (1-(1-p)**ks)*100, 'o-', lw=2, label=f'pass@k (p={p})')
    ax.plot(ks, (p**ks)*100, 's--', lw=2, label=f'consistency p^k (p={p})')
ax.set_xlabel('k attempts'); ax.set_ylabel('%'); ax.set_title('pass@k rises, consistency falls')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Validate: pass@k rises while consistency falls

Two opposite curves from the same per-attempt success $p$. pass@k $=1-(1-p)^k$
increases toward 1 (any success becomes likely); consistency $p^k$ decreases toward 0
(all-success becomes unlikely). We confirm the monotonic directions.

In [ ]:
p = 0.4
passk = [1 - (1-p)**k for k in ks]
cons  = [p**k for k in ks]
print(f'{"k":>2} {"pass@k":>8} {"consistency":>12}')
for k, pk, ck in zip(ks, passk, cons):
    print(f'{k:>2} {pk:>8.3f} {ck:>12.3f}')
assert all(passk[i] < passk[i+1] for i in range(len(passk)-1)), 'pass@k rises with k'
assert all(cons[i] > cons[i+1] for i in range(len(cons)-1)), 'consistency falls with k'
print('\n✅ more attempts help pass@k but hurt consistency — report the one that matches your use case')

## 3. Estimate pass@k by Monte Carlo

Run many independent attempts and estimate the empirical pass@k — it should track $1-(1-p)^k$.

In [ ]:
def empirical_pass_at_k(p, k, trials=20000, seed=0):
    g = np.random.default_rng(seed)
    outcomes = g.random((trials, k)) < p
    return outcomes.any(axis=1).mean()

for k in [1, 3, 5]:
    emp = empirical_pass_at_k(0.4, k, seed=k)
    theo = 1-(1-0.4)**k
    print(f'k={k}: empirical {emp:.3f}  theory {theo:.3f}')

### Validate: the closed-form pass@k matches Monte-Carlo

The formula $1-(1-p)^k$ should match an empirical estimate from actually simulating $k$
independent attempts many times. Agreeing to ~1% confirms the metric is what the
simulation measures.

In [ ]:
for k in [1, 3, 5]:
    emp = empirical_pass_at_k(0.4, k, seed=k)
    theo = 1 - (1 - 0.4)**k
    print(f'k={k}: empirical {emp:.3f}  theory {theo:.3f}  |diff| {abs(emp-theo):.3f}')
    assert abs(emp - theo) < 0.02, 'empirical pass@k must match the closed form'
print('\n✅ pass@k = 1-(1-p)^k, confirmed by simulation')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **single-run evaluation** | stochastic agents need many runs; one number is noise (demo) |
| **outcome-only scoring** | ignores wasteful trajectories that cost 10× for the same answer |
| **pass@k hides cost** | k attempts cost k×; high pass@k can be economically useless |
| **benchmark contamination** | tasks leaking into training inflate scores |
| **reward/judge gaming** | agents exploit weak automated graders; audit the grader |

Demo: a single run is a coin flip; many-run means are tight.

In [ ]:
# Why a SINGLE run lies: with p=0.5, one run is a coin flip — you cannot distinguish a
# good agent from a bad one. We show the variance of a 1-run estimate vs a 20-run mean.
p = 0.5
one_run = [float(np.random.default_rng(s).random() < p) for s in range(200)]
twenty  = [np.mean(np.random.default_rng(s).random(20) < p) for s in range(200)]
print(f'1-run estimate : mean {np.mean(one_run):.2f}, std {np.std(one_run):.2f}  (0 or 1 — useless)')
print(f'20-run estimate: mean {np.mean(twenty):.2f}, std {np.std(twenty):.2f}  (tight around 0.5)')
assert np.std(twenty) < np.std(one_run), 'averaging many runs shrinks the variance'
print('\nA single agent run is a coin flip; only many-run metrics (pass@k, mean±CI) are trustworthy.')

## ✏️ Your turn — pass@k

Implement `pass_at_k(p, k)` = probability of at least one success in `k` independent attempts.

In [ ]:
def pass_at_k(p, k):
    """TODO(you): return 1 - (1 - p)**k."""
    # TODO
    return ...


In [ ]:
print('pass@1 (p=0.4):', round(pass_at_k(0.4, 1), 3), '(expected 0.4)')
print('pass@5 (p=0.4):', round(pass_at_k(0.4, 5), 3), '(expected ~0.922)')
assert abs(pass_at_k(0.4, 1) - 0.4) < 1e-9
assert abs(pass_at_k(0.4, 5) - (1-0.6**5)) < 1e-9
assert abs(pass_at_k(0.0, 10) - 0.0) < 1e-9
assert abs(pass_at_k(1.0, 3) - 1.0) < 1e-9
print('\n✅ pass@k matches the closed form.')

<details>
<summary>Solution</summary>

```python
def pass_at_k(p, k):
    return 1 - (1 - p) ** k
```

Report pass@k *and* consistency: high pass@k with low consistency means capable-but-unreliable — not production-ready. And remember cost/latency budgets gate whether even a reliable agent ships.
</details>

## Recap

- Score **outcome** and **trajectory** (efficiency = opt/steps).
- A single stochastic run is noise; report **pass@k** (capability) and **consistency** (reliability).
- Monte-Carlo estimates track the closed form $1-(1-p)^k$.
- LLM-as-judge fills the gaps reference metrics can't — but validate the judge first.

## Key takeaways

- **Agents are stochastic — one run lies.** Evaluate over many runs; a single-run
  estimate has huge variance (demo).
- **pass@k rises, consistency falls** with $k$ (verified) — pick the metric matching
  your product (any-success vs every-success).
- **pass@k $=1-(1-p)^k$** matches simulation (verified) — a cheap way to reason about
  retry budgets.
- **Score outcome *and* trajectory:** solving the task matters, but so do the steps and
  cost it took.